# Déploiement fidèle sur SpiNNaker (SANDS)

Ce notebook lance `spinnaker_faithful_eval.py` dans le noyau Jupyter, là où `pyNN.spiNNaker` est installé.
Exécutez les cellules dans l'ordre. Le notebook peut être placé dans `work/` ou dans `work/SpiNNaker/`.

## 1. Dossier de travail et dépendances

In [ ]:
import importlib
import os
import site
import subprocess
import sys

# Se placer dans le dossier qui contient spinnaker_plan.json (work/).
if not os.path.exists("spinnaker_plan.json") and os.path.exists("../spinnaker_plan.json"):
    os.chdir("..")
assert os.path.exists("spinnaker_plan.json"), "spinnaker_plan.json introuvable : extraire sands_bundle.tar.gz dans work/"
print("Dossier de travail :", os.getcwd())


def pip_install(*packages):
    """Installe dans l'environnement du noyau (et non dans celui du terminal)."""
    command = [sys.executable, "-m", "pip", "install", "--quiet", *packages]
    if subprocess.call(command) != 0:
        subprocess.check_call(command[:4] + ["--user"] + command[4:])
    importlib.invalidate_caches()
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


try:
    import torch
except ImportError:
    pip_install("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
try:
    import spikingjelly
except ImportError:
    # --no-deps : seuls torch et numpy sont nécessaires ici, inutile de retélécharger torch.
    pip_install("--no-deps", "spikingjelly")

## 2. Vérification de l'environnement

In [ ]:
import pyNN.spiNNaker as sim
import spikingjelly
import torch

sys.path.insert(0, os.path.join(os.getcwd(), "SpiNNaker"))
import spinnaker_energy
import spinnaker_faithful_eval as faithful

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("pyNN        :", sim.__name__, "OK")
print("neurone     :", faithful.delta_neuron_model(sim).__name__)
print("script      :", "à jour" if hasattr(faithful, "ring_buffer_shift") else "ANCIENNE VERSION, à remplacer")

import configparser

config = configparser.ConfigParser(strict=False)
config.read([os.path.expanduser("~/.spynnaker.cfg"), "spynnaker.cfg"])
print("énergie     :", config.get("Reports", "write_energy_report", fallback="False"), "(doit valoir True)")

## 3. Test sur un seul modèle

La ligne finale doit indiquer `SUCCESS`, avec `carte` proche de `virgule_fixe`.

In [ ]:
faithful.main(["--dataset", "smnist", "--limit", "1"])

## 4. Tous les modèles déployables

Du plus léger au plus lourd. La cellule peut être relancée : les runs déjà réussis sont sautés.
Si `carte` est nettement sous `virgule_fixe`, ajouter `"--time-scale-factor", "50"` à la liste d'arguments.

In [ ]:
for dataset in ["smnist", "shd", "ssc", "nmnist"]:
    faithful.main(["--dataset", dataset])

## 5. Précision : PyTorch, émulation, carte

In [ ]:
import csv

with open("results_spinnaker_faithful/summary_benchmark.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"{'run':<42} {'statut':<15} {'pytorch':>8} {'virg. fixe':>10} {'carte':>7}")
for row in rows:
    print(
        f"{row['run_id']:<42} {row['status']:<15} {row['accuracy_pytorch_pct']:>8} "
        f"{row['accuracy_fixed_point_pct']:>10} {row['accuracy_pct']:>7}"
    )

## 6. Énergie réelle et tableau final

Écrit `results_spinnaker_faithful/spinnaker_energy.csv` et, si `theoretical_energy_results.csv` est présent, `final_energy_results.csv`.

In [ ]:
spinnaker_energy.main(["--results-dir", "results_spinnaker_faithful"])